# 生成モデルの全体像

生成モデルの仕事は、入力にラベルを付けることではなく、データが生まれる確率的な仕組みを近似し、まだ観測していないサンプルを作れるようにすることです。画像なら画素の並び、文章ならトークン列、音声なら波形やスペクトログラムが対象になります。中心にある問いはいつも同じです。どのサンプルが自然で、どのサンプルが稀で、どの失敗が危険なのかを確率分布として扱えるかです。

## 小さな世界で分布を見る

4個の0/1値からなるパターンだけを扱います。空間全体は16通りしかないため、経験分布、独立近似、自己回帰近似、崩壊した生成器を同じ土俵で比較できます。高次元データでも、考えている対象は「あり得るサンプル全体のうち、どこに質量を置くか」です。

In [ ]:
import itertools
import math
import random
from collections import Counter, defaultdict
from statistics import mean

random.seed(7)

patterns = list(itertools.product([0, 1], repeat=4))
toy_data = [
    (1, 1, 1, 0), (1, 1, 1, 0), (1, 1, 1, 0),
    (1, 1, 0, 0), (1, 1, 0, 0),
    (1, 0, 0, 0),
    (0, 0, 0, 1), (0, 0, 1, 1),
]

counts = Counter(toy_data)
empirical = {p: counts[p] / len(toy_data) for p in patterns}

for p, prob in sorted(empirical.items(), key=lambda kv: (-kv[1], kv[0])):
    if prob > 0:
        print(p, 'prob=', round(prob, 3))

経験分布は観測データをそのまま確率表にしたものです。観測されたパターンには質量を置けますが、未観測のパターンには確率0を置いてしまいます。これは過学習の極端な形でもあります。サンプル数が少ないと、まだ見ていない自然なパターンまで不可能扱いしてしまいます。

In [ ]:
def sample_from_distribution(dist, n):
    keys = list(dist.keys())
    weights = [dist[k] for k in keys]
    out = []
    for _ in range(n):
        r = random.random() * sum(weights)
        total = 0.0
        for k, w in zip(keys, weights):
            total += w
            if r <= total:
                out.append(k)
                break
    return out

print(sample_from_distribution(empirical, 10))

## 独立近似の強みと限界

最も単純な近似は、4つの位置が互いに独立に1になると仮定する方法です。各位置の1の割合だけを覚えればよいので学習は安定します。一方で「左側が1なら右側も1になりやすい」といった結び付きは表せません。

In [ ]:
def fit_independent_bernoulli(data, alpha=1.0):
    n = len(data)
    d = len(data[0])
    return [(sum(x[j] for x in data) + alpha) / (n + 2 * alpha) for j in range(d)]


def prob_independent(pattern, probs):
    p = 1.0
    for value, q in zip(pattern, probs):
        p *= q if value == 1 else (1 - q)
    return p


bernoulli_probs = fit_independent_bernoulli(toy_data, alpha=0.5)
independent = {p: prob_independent(p, bernoulli_probs) for p in patterns}

print('pixel probabilities:', [round(p, 3) for p in bernoulli_probs])
for p, prob in sorted(independent.items(), key=lambda kv: -kv[1])[:6]:
    print(p, 'prob=', round(prob, 3), 'observed=', counts[p])

独立モデルは未観測パターンにも確率を配るため、経験分布より滑らかです。しかし、位置間の相関を持てないため、実データで強く出る形と、部品だけを寄せ集めた形を区別しにくくなります。生成モデルでは、滑らかに一般化する力と、データの構造を壊さない表現力の両方が必要になります。

## 自己回帰分解で依存関係を持つ

任意の同時分布は、左から順に条件付き確率を掛け合わせる形に分解できます。文章生成ではこの考え方がそのまま使われます。過去のトークンを条件に次のトークンを出すため、依存関係を自然に表せます。

In [ ]:
def fit_autoregressive_binary(data, alpha=0.5):
    tables = [defaultdict(lambda: [alpha, alpha]) for _ in range(len(data[0]))]
    for x in data:
        prefix = ()
        for j, value in enumerate(x):
            tables[j][prefix][value] += 1.0
            prefix = prefix + (value,)
    return tables


def prob_autoregressive(pattern, tables):
    prefix = ()
    p = 1.0
    for j, value in enumerate(pattern):
        zero, one = tables[j][prefix]
        total = zero + one
        p *= (one if value == 1 else zero) / total
        prefix = prefix + (value,)
    return p


def sample_autoregressive(tables, n):
    out = []
    for _ in range(n):
        prefix = ()
        sample = []
        for j in range(len(tables)):
            zero, one = tables[j][prefix]
            q = one / (zero + one)
            value = 1 if random.random() < q else 0
            sample.append(value)
            prefix = prefix + (value,)
        out.append(tuple(sample))
    return out

ar_tables = fit_autoregressive_binary(toy_data, alpha=0.5)
autoregressive = {p: prob_autoregressive(p, ar_tables) for p in patterns}

for p, prob in sorted(autoregressive.items(), key=lambda kv: -kv[1])[:6]:
    print(p, 'prob=', round(prob, 3), 'observed=', counts[p])
print('samples:', sample_autoregressive(ar_tables, 8))

自己回帰モデルは依存関係を表しやすい反面、生成時に1ステップずつ進むため並列化しにくくなります。明示的な尤度を計算しやすいこと、教師あり学習に近い形で訓練しやすいこと、長い系列では生成が遅くなりやすいことが基本的なトレードオフです。

## 潜在変数は見えない原因を表す

混合分布やVAEでは、観測された x の背後に潜在変数 z を置きます。z は「どのクラスタか」「どの書き方か」「どの姿勢か」のような、サンプルの違いを生む内部要因です。潜在変数を持つと、多峰性や連続的な変化を扱いやすくなります。

In [ ]:
def gaussian_pdf(x, mu, sigma):
    return math.exp(-0.5 * ((x - mu) / sigma) ** 2) / (math.sqrt(2 * math.pi) * sigma)


def mixture_pdf(x, weights, means, sigmas):
    return sum(w * gaussian_pdf(x, m, s) for w, m, s in zip(weights, means, sigmas))

xs = [-4, -3, -2, -1, 0, 1, 2, 3, 4]
single = [gaussian_pdf(x, 0.0, 2.2) for x in xs]
mixture = [mixture_pdf(x, [0.55, 0.45], [-2.0, 2.2], [0.6, 0.7]) for x in xs]

print('x | single | mixture')
for x, a, b in zip(xs, single, mixture):
    print(f'{x:>2} | {a:.4f} | {b:.4f}')

単一のガウス分布は山を1つしか持てません。混合分布は潜在的な成分を使うことで、離れた複数の山を表せます。VAEはこの発想をニューラルネットワークで拡張し、連続潜在変数から複雑なデータを復元する写像を学びます。

## フローは可逆変換で密度を追跡する

正規化フローは、単純な分布から始めて、可逆な変換を重ねて複雑な分布を作ります。可逆なので、サンプル生成だけでなく密度計算もできます。密度の変化はヤコビアンの行列式で補正します。

In [ ]:
def affine_flow_forward(z, scale, shift):
    x = scale * z + shift
    log_abs_det = math.log(abs(scale))
    return x, log_abs_det


def standard_normal_logpdf(z):
    return -0.5 * z * z - 0.5 * math.log(2 * math.pi)

scale, shift = 1.8, -0.7
for z in [-1.5, 0.0, 1.5]:
    x, log_det = affine_flow_forward(z, scale, shift)
    log_px = standard_normal_logpdf(z) - log_det
    print('z=', z, 'x=', round(x, 3), 'log p(x)=', round(log_px, 3))

フローは尤度を正確に計算しやすい一方、変換を可逆に保つ制約があります。画像のような高次元データでは、この制約が表現力や設計の複雑さに跳ね返ります。

## GANは識別器との競争で生成器を磨く

GANは密度を直接書かず、生成器と識別器を競わせます。生成器は偽物を作り、識別器は本物と偽物を見分けます。うまく進むと鋭いサンプルを作れますが、訓練は不安定になりやすく、多様性が落ちる失敗も起きます。

In [ ]:
def diversity(samples):
    return len(set(samples)) / len(samples)

regular_samples = sample_autoregressive(ar_tables, 200)
collapsed_samples = [(1, 1, 1, 0) for _ in range(200)]

print('regular unique ratio  =', round(diversity(regular_samples), 3))
print('collapsed unique ratio=', round(diversity(collapsed_samples), 3))
print('regular unique count  =', len(set(regular_samples)))
print('collapsed unique count=', len(set(collapsed_samples)))

多様性が落ちると、見た目の品質が高くても用途によっては失敗です。推薦、シミュレーション、データ拡張では、頻出パターンだけを繰り返す生成器は価値が低くなります。品質と多様性を同時に見る習慣が重要です。

## 拡散モデルは壊し方を学んで戻す

拡散モデルは、データに段階的にノイズを加える前向き過程と、ノイズからデータへ戻す逆向き過程を持ちます。学習では、ある時刻で混ざったノイズやスコアを予測します。生成ではノイズから始めて、少しずつ自然なサンプルへ近づけます。

In [ ]:
x0 = 2.0
schedule = [0.08, 0.16, 0.28, 0.40]
trajectory = [x0]
noises = []

x = x0
for beta in schedule:
    eps = random.gauss(0, 1)
    noises.append(eps)
    x = math.sqrt(1 - beta) * x + math.sqrt(beta) * eps
    trajectory.append(x)

oracle = trajectory[-1]
for beta, eps in zip(reversed(schedule), reversed(noises)):
    oracle = (oracle - math.sqrt(beta) * eps) / math.sqrt(1 - beta)

naive = trajectory[-1]
for beta in reversed(schedule):
    naive = naive / math.sqrt(1 - beta)

print('forward:', [round(v, 3) for v in trajectory])
print('oracle reverse:', round(oracle, 3))
print('naive reverse :', round(naive, 3))

逆向きで必要なのは、どんなノイズが混ざったかを推定する力です。実際のモデルは oracle のように真のノイズを知りません。ニューラルネットワークがその推定器になり、推定誤差が小さいほど自然なサンプルへ戻しやすくなります。

## 評価は尤度・品質・多様性・制御性を分ける

生成モデルの良し悪しは1つの数字に圧縮できません。尤度が高いモデルは確率校正に強いことがあります。見た目の品質が高いモデルは人間評価や下流タスクで有利なことがあります。多様性が低いモデルは、平均的には良く見えても実運用で同じ出力を繰り返します。潜在空間を制御したい用途では、補間や属性編集のしやすさも重要になります。

In [ ]:
def nll(dist, data, eps=1e-12):
    return mean([-math.log(max(dist[x], eps)) for x in data])


def coverage(samples):
    return len(set(samples)) / len(patterns)

models = {
    'empirical': empirical,
    'independent': independent,
    'autoregressive': autoregressive,
}

for name, dist in models.items():
    samples = sample_from_distribution(dist, 500)
    print(name)
    print('  train NLL =', round(nll(dist, toy_data), 3))
    print('  coverage  =', round(coverage(samples), 3))
    print('  diversity =', round(diversity(samples), 3))

経験分布は訓練データへの尤度が高くなりますが、未観測パターンへの広がりは弱くなります。独立モデルは滑らかに広がりますが、構造を落とします。自己回帰モデルは依存関係を持ちながら尤度も計算できます。どれが最良かは、データ量、必要な生成速度、品質、多様性、密度評価の必要性で変わります。

## 手法を選ぶ軸

明示的な尤度が必要なら、自己回帰モデルやフローが候補になります。潜在表現を操作したいなら、VAE系や表現学習を組み合わせたモデルが扱いやすくなります。最高品質の画像生成を狙うなら、拡散モデルが強力です。高速サンプリングが最優先なら、GANや蒸留された拡散モデルが候補になります。重要なのは、流行ではなく制約から設計を選ぶことです。

In [ ]:
def recommend_family(explicit_likelihood, fast_sampling, latent_control, top_visual_quality, stable_training):
    if explicit_likelihood and fast_sampling:
        return 'flow or compact autoregressive model'
    if explicit_likelihood:
        return 'autoregressive model or flow'
    if latent_control and stable_training:
        return 'VAE-family or latent diffusion with encoder'
    if top_visual_quality and not fast_sampling:
        return 'diffusion-family'
    if fast_sampling:
        return 'GAN-family, flow, or distilled diffusion'
    return 'hybrid design matched to data and constraints'

cases = [
    dict(explicit_likelihood=True, fast_sampling=False, latent_control=False, top_visual_quality=False, stable_training=True),
    dict(explicit_likelihood=False, fast_sampling=False, latent_control=True, top_visual_quality=False, stable_training=True),
    dict(explicit_likelihood=False, fast_sampling=False, latent_control=False, top_visual_quality=True, stable_training=True),
    dict(explicit_likelihood=False, fast_sampling=True, latent_control=False, top_visual_quality=True, stable_training=False),
]

for case in cases:
    print(case, '->', recommend_family(**case))

生成モデルは、確率分布をどう表し、どうサンプルを引き、どの失敗を避けるかの設計です。小さな0/1分布でも、過学習、独立仮定、依存関係、潜在変数、多様性低下、評価の分離はすべて現れます。大規模な画像生成や言語生成でも、見ている問題は同じです。